# Music Annotation Project V7.5: Exact Observed-Geometry Preservation Benchmark

**Runtime:** CPU. Select **Runtime > Run all**.

V7.5 tests a document-preservation architecture rather than unrestricted image reconstruction.

## Governing rule

Within the dilation-2 annotation-removal mask:

1. observed pixels classified as confidently printed musical geometry are copied **byte-for-byte**;
2. confidently light/grey annotation pixels are replaced with locally estimated paper;
3. only pixels assigned to the explicit ambiguous-overlap class may use Telea;
4. all pixels outside the removal mask remain byte-for-byte unchanged.

A genuine pencil/print overlap cannot be identified with certainty from an annotated-only scan. Therefore V7.5 does not claim to detect genuine overlap directly. It benchmarks a conservative observable proxy and exports the ambiguity map for review.

## Methods

- `telea`: cached V7.1 dilation-2 baseline;
- `hard_raw_t80`: direct raw-intensity comparator;
- `hard_raw_t110`: direct raw-intensity comparator;
- `geometry_t80`: exact geometry preservation, paper whitening, Telea only in ambiguous pixels;
- `geometry_t110`: stronger geometry candidate using the same architecture.

## Geometry evidence

The candidate uses only the observed page and mask. It combines:

- long horizontal dark runs supported on both sides of the mask, for stave and other printed horizontal lines;
- narrow vertical dark runs, for stems and bar lines;
- compact dark components connected to observed notation outside the removal mask, for noteheads and small glyph structure.

Dark pixels are **not** automatically preserved. They must satisfy geometry and continuity tests.

Clean targets are used only after reconstruction for scoring and diagnostic confusion tables.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless scikit-image pandas matplotlib


## Configuration and controlled-page discovery


In [ ]:
import cv2, json, hashlib, shutil, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from skimage.metrics import structural_similarity as ssim

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'; OBSERVED_DIR=ROOT/'aligned'; CORRECTED_MASK_DIR=ROOT/'masks'/'corrected'
V71_ROOT=ROOT/'reconstruction_benchmark_v7_1_real'; V71_STATE=V71_ROOT/'benchmark_state.json'
OUTPUT_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'; CACHE=OUTPUT_ROOT/'persistent_cache'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v7_5'; OUT=OUTPUT_ROOT/'runs'/RUN
TABLES=OUT/'tables'; CONTACTS=OUT/'contact_sheets'; CROPS=OUT/'failure_crops'; MAPS=OUT/'classification_maps'
LOCAL=Path('/content/reconstruction_geometry_v7_5'); LCLEAN=LOCAL/'clean'; LOBS=LOCAL/'observed'
for d in [OUTPUT_ROOT,CACHE,OUT,TABLES,CONTACTS,CROPS,MAPS,LOCAL,LCLEAN,LOBS]: d.mkdir(parents=True,exist_ok=True)

DILATION=2
EXCLUDE_PAGES={'Page201'}  # known stale/incorrect historical mask; empty corrected masks are also skipped
RAW_THRESHOLDS=[80,110]
PAPER_THRESHOLD=155
NORMALISED_PAPER_THRESHOLD=165
BACKGROUND_SIGMA=75.0
HORIZONTAL_LENGTH=35
VERTICAL_LENGTH=9
GEOMETRY_GROW=1
OUTSIDE_SUPPORT_RADIUS=13
COMPACT_MIN_AREA=6
COMPACT_MAX_AREA=500
COMPACT_MAX_DIMENSION=42
COMPACT_MIN_FILL=0.16
AMBIGUOUS_DARK_LOW=65
CROP_MARGIN=48
BOOTSTRAP_SAMPLES=2000
MAX_CONTACT_PAGES=24
MAX_FAILURE_CROPS=20
FORCE_REBUILD=False
METHODS=['telea','hard_raw_t80','hard_raw_t110','geometry_t80','geometry_t110']
SCHEMA='v7.5_exact_observed_geometry_preservation'

def image_files(folder):
    folder=Path(folder); return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def maps(folder): return {p.stem:p for p in image_files(folder)}
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def object_hash(value): return hashlib.md5(json.dumps(value,sort_keys=True).encode()).hexdigest()
def read_json(path,default=None):
    try:return json.loads(Path(path).read_text()) if Path(path).exists() else default
    except:return default
def valid_npz(path,keys):
    try:
        with np.load(path) as z:return set(keys).issubset(z.files)
    except:return False
def copy_if_changed(source,target):
    if not target.exists() or md5(source)!=md5(target): shutil.copy2(source,target)
def find_local(folder,page):
    hits=[p for p in image_files(folder) if p.stem.lower()==page.lower()]; return hits[0] if len(hits)==1 else None
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0]
    assert np.array_equal(out[mask==0],original[mask==0]);return out

state=read_json(V71_STATE)
if not state or not isinstance(state.get('items'),dict): raise RuntimeError(f'Invalid V7.1 state: {V71_STATE}')
clean_map,observed_map,corrected_map=maps(CLEAN_DIR),maps(OBSERVED_DIR),maps(CORRECTED_MASK_DIR)
pages=[];issues=[];excluded=[]
for key,item in sorted(state['items'].items()):
    if int(item.get('dilation',-1))!=DILATION:continue
    page=item.get('page');cache=Path(item.get('reconstruction_cache',''))
    if page in EXCLUDE_PAGES:excluded.append({'page':page,'reason':'explicit exclusion'});continue
    if page not in clean_map or page not in observed_map or page not in corrected_map:
        issues.append({'page':page,'reason':'clean, observed or corrected mask missing'});continue
    corrected=cv2.imread(str(corrected_map[page]),0)
    if corrected is None:issues.append({'page':page,'reason':'corrected mask unreadable'});continue
    if not np.any(corrected>0):excluded.append({'page':page,'reason':'empty corrected hard-negative mask'});continue
    if not item.get('reconstruction_complete') or not valid_npz(cache,['mask','telea']):
        issues.append({'page':page,'reason':'valid V7.1 dilation-2 cache missing'});continue
    clean=cv2.imread(str(clean_map[page]),0);observed=cv2.imread(str(observed_map[page]),0)
    with np.load(cache) as z:shape=z['mask'].shape
    if clean is None or observed is None or clean.shape!=observed.shape or shape!=observed.shape:
        issues.append({'page':page,'reason':'shape mismatch'});continue
    copy_if_changed(clean_map[page],LCLEAN/(page+clean_map[page].suffix.lower()))
    copy_if_changed(observed_map[page],LOBS/(page+observed_map[page].suffix.lower()));pages.append(page)
pages=sorted(set(pages));pd.DataFrame(issues).to_csv(TABLES/'discovery_issues.csv',index=False);pd.DataFrame(excluded).to_csv(TABLES/'excluded_pages.csv',index=False)
if not pages:raise RuntimeError('No eligible controlled pages.')
print('Eligible pages:',len(pages),'Excluded:',excluded,'Issues:',len(issues))


## Geometry classifier and reconstruction methods


In [ ]:
def background_normalise(image):
    bg=cv2.GaussianBlur(image,(0,0),BACKGROUND_SIGMA);return cv2.divide(image,np.maximum(bg,1),scale=255)
def paper_map(observed,mask):
    result=np.full_like(observed,255);n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
    for i in range(1,n):
        comp=labels==i;ring=cv2.dilate(comp.astype(np.uint8),np.ones((21,21),np.uint8))>0;ring&=~comp
        values=observed[ring&(observed>=180)];result[comp]=int(np.median(values)) if values.size else 255
    return result

def supported_horizontal(observed,mask,dark_threshold):
    dark=observed<dark_threshold;outside=dark&(mask==0)
    outside_lines=cv2.morphologyEx(outside.astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,HORIZONTAL_LENGTH),np.uint8))>0
    candidate_lines=cv2.morphologyEx(dark.astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,HORIZONTAL_LENGTH),np.uint8))>0
    support=cv2.dilate(outside_lines.astype(np.uint8),np.ones((3,2*OUTSIDE_SUPPORT_RADIUS+1),np.uint8))>0
    return candidate_lines&support&(mask>0)

def supported_vertical(observed,mask,dark_threshold):
    dark=observed<dark_threshold;outside=dark&(mask==0)
    outside_lines=cv2.morphologyEx(outside.astype(np.uint8),cv2.MORPH_OPEN,np.ones((VERTICAL_LENGTH,1),np.uint8))>0
    candidate_lines=cv2.morphologyEx(dark.astype(np.uint8),cv2.MORPH_OPEN,np.ones((VERTICAL_LENGTH,1),np.uint8))>0
    support=cv2.dilate(outside_lines.astype(np.uint8),np.ones((2*OUTSIDE_SUPPORT_RADIUS+1,3),np.uint8))>0
    return candidate_lines&support&(mask>0)

def supported_compact(observed,mask,dark_threshold):
    dark=observed<dark_threshold;n,labels,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);result=np.zeros_like(dark)
    outside_near=cv2.dilate(((dark)&(mask==0)).astype(np.uint8),np.ones((5,5),np.uint8))>0
    for i in range(1,n):
        x,y,w,h,area=map(int,stats[i]);fill=area/max(w*h,1);comp=labels==i
        if area<COMPACT_MIN_AREA or area>COMPACT_MAX_AREA or max(w,h)>COMPACT_MAX_DIMENSION or fill<COMPACT_MIN_FILL:continue
        inside=comp&(mask>0)
        if inside.any() and np.any(comp&outside_near):result|=inside
    return result

def geometry_evidence(observed,mask,dark_threshold):
    horizontal=supported_horizontal(observed,mask,dark_threshold)
    vertical=supported_vertical(observed,mask,dark_threshold)
    compact=supported_compact(observed,mask,dark_threshold)
    geometry=horizontal|vertical|compact
    if GEOMETRY_GROW>0:
        geometry=cv2.dilate(geometry.astype(np.uint8),np.ones((2*GEOMETRY_GROW+1,2*GEOMETRY_GROW+1),np.uint8))>0
        geometry&=(observed<dark_threshold)&(mask>0)
    return geometry,horizontal,vertical,compact

def hard_raw(observed,mask,paper,threshold):
    out=observed.copy();whiten=(mask>0)&(observed>=threshold);out[whiten]=paper[whiten];return immutable(observed,out,mask)

def geometry_candidate(observed,mask,telea,paper,norm,dark_threshold):
    preserve,horizontal,vertical,compact=geometry_evidence(observed,mask,dark_threshold)
    remove=(mask>0)&~preserve&(observed>=PAPER_THRESHOLD)&(norm>=NORMALISED_PAPER_THRESHOLD)
    # Dark non-geometric pixels and midtones are uncertain. Only this class may be reconstructed.
    ambiguous=(mask>0)&~preserve&~remove
    out=observed.copy();out[remove]=paper[remove];out[ambiguous]=telea[ambiguous]
    out[preserve]=observed[preserve]
    assert np.array_equal(out[preserve],observed[preserve])
    return immutable(observed,out,mask),preserve,remove,ambiguous,horizontal,vertical,compact


## Candidate generation and persistent cache


In [ ]:
cache_records=[];route_rows=[]
for index,page in enumerate(pages,1):
    observed=cv2.imread(str(find_local(LOBS,page)),0);item=state['items'][f'{page}|d{DILATION}']
    with np.load(item['reconstruction_cache']) as z:mask=z['mask'];telea=z['telea']
    fingerprint=object_hash({'schema':SCHEMA,'v71':item.get('fingerprint'),'observed':md5(find_local(LOBS,page)),
        'settings':{'raw':RAW_THRESHOLDS,'paper':[PAPER_THRESHOLD,NORMALISED_PAPER_THRESHOLD],'sigma':BACKGROUND_SIGMA,
        'horizontal':HORIZONTAL_LENGTH,'vertical':VERTICAL_LENGTH,'grow':GEOMETRY_GROW,'support':OUTSIDE_SUPPORT_RADIUS,
        'compact':[COMPACT_MIN_AREA,COMPACT_MAX_AREA,COMPACT_MAX_DIMENSION,COMPACT_MIN_FILL]}})
    cache_path=CACHE/f'{page}_{fingerprint}.npz'
    required=METHODS+['mask','g80_preserve','g80_remove','g80_ambiguous','g110_preserve','g110_remove','g110_ambiguous']
    if cache_path.exists() and not FORCE_REBUILD and valid_npz(cache_path,required):hit=True
    else:
        paper=paper_map(observed,mask);norm=background_normalise(observed)
        outputs={'telea':telea,'hard_raw_t80':hard_raw(observed,mask,paper,80),'hard_raw_t110':hard_raw(observed,mask,paper,110)};extra={}
        for threshold in RAW_THRESHOLDS:
            result,preserve,remove,ambiguous,horizontal,vertical,compact=geometry_candidate(observed,mask,telea,paper,norm,threshold)
            outputs[f'geometry_t{threshold}']=result
            for name,value in [('preserve',preserve),('remove',remove),('ambiguous',ambiguous),('horizontal',horizontal),('vertical',vertical),('compact',compact)]:extra[f'g{threshold}_{name}']=value.astype(np.uint8)
        for method,result in outputs.items():assert np.array_equal(result[mask==0],observed[mask==0])
        np.savez_compressed(cache_path,mask=mask,**outputs,**extra);hit=False
    with np.load(cache_path) as z:
        for threshold in RAW_THRESHOLDS:
            preserve=z[f'g{threshold}_preserve']>0;remove=z[f'g{threshold}_remove']>0;ambiguous=z[f'g{threshold}_ambiguous']>0;m=z['mask']>0
            route_rows.append({'page':page,'method':f'geometry_t{threshold}','mask_pixels':int(m.sum()),'preserve_pixels':int(preserve.sum()),'remove_pixels':int(remove.sum()),'ambiguous_pixels':int(ambiguous.sum()),
                'preserve_fraction':float(preserve.sum()/max(m.sum(),1)),'remove_fraction':float(remove.sum()/max(m.sum(),1)),'ambiguous_fraction':float(ambiguous.sum()/max(m.sum(),1)),
                'partition_exact':bool(np.array_equal((preserve|remove|ambiguous),m) and not np.any((preserve&remove)|(preserve&ambiguous)|(remove&ambiguous)))})
    cache_records.append({'page':page,'cache':str(cache_path),'cache_hit':hit});print(f'{index}/{len(pages)} {page}:', 'HIT' if hit else 'MISS')
pd.DataFrame(cache_records).to_csv(TABLES/'cache_status.csv',index=False);pd.DataFrame(route_rows).to_csv(TABLES/'routing_by_page.csv',index=False)


## Structure-first evaluation


In [ ]:
def printed_ink(image):return cv2.adaptiveThreshold(image,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(image):return cv2.Canny(image,70,170)>0
def classify(clean,mask):
    ink=printed_ink(clean);long=cv2.morphologyEx(ink.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0;near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0;m=mask>0
    return {'all':m,'background':m&~ink,'staff':m&staff,'symbol':m&ink&~near,'mixed':m&ink&near},ink,staff

def tolerance_bundle(reference,prediction,tol=1):
    kernel=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(reference.astype(np.uint8),kernel)>0;pdil=cv2.dilate(prediction.astype(np.uint8),kernel)>0
    return {'reference':reference,'prediction':prediction,'prediction_match':prediction&rd,'reference_match':reference&pdil}
def f1_from_bundle(bundle,region):
    rc=int(np.sum(bundle['reference']&region));pc=int(np.sum(bundle['prediction']&region))
    if not rc:return np.nan,np.nan,np.nan
    precision=float(np.sum(bundle['prediction_match']&region)/max(pc,1));recall=float(np.sum(bundle['reference_match']&region)/rc)
    return precision,recall,float(2*precision*recall/max(precision+recall,1e-12))
def method_bundle(clean,recon,clean_ink,clean_edges):
    diff=clean.astype(np.float32)-recon.astype(np.float32);_,sm=ssim(clean,recon,data_range=255,full=True);ri=printed_ink(recon);re=edges(recon)
    return {'abs':np.abs(diff),'sq':diff*diff,'ssim':sm.astype(np.float32,copy=False),'ink':tolerance_bundle(clean_ink,ri),'edge':tolerance_bundle(clean_edges,re),
        'recon_ink':ri,'dark_smudge':(recon<180)&(clean>=220),'clean_light':clean>=220}
def staff_geometry(clean_ink,recon_ink,staff_region):
    ref=clean_ink&staff_region;pred=recon_ink&staff_region;cols=np.where(ref.sum(axis=0)>0)[0];offsets=[];thickness=[];gaps=valid=0
    for x in cols:
        ry=np.where(ref[:,x])[0]
        if not len(ry):continue
        py=np.where(pred[:,x])[0];valid+=1
        if not len(py):gaps+=1;continue
        centre=float(np.median(ry));nearest=float(py[np.argmin(np.abs(py-centre))]);offsets.append(abs(nearest-centre));thickness.append(abs(len(py)-len(ry)))
    return {'staff_gap_rate':float(gaps/max(valid,1)),'staff_centreline_mae':float(np.mean(offsets)) if offsets else np.nan,'staff_centreline_p95':float(np.percentile(offsets,95)) if offsets else np.nan,'staff_thickness_mae':float(np.mean(thickness)) if thickness else np.nan}
def score(bundle,region,clean_ink=None,staff_region=None):
    n=int(region.sum())
    if not n:return None
    mse=float(np.mean(bundle['sq'][region]));ip,ir,if1=f1_from_bundle(bundle['ink'],region);_,_,ef=f1_from_bundle(bundle['edge'],region)
    out={'pixels':n,'mae':float(np.mean(bundle['abs'][region])),'psnr':float(20*np.log10(255/math.sqrt(max(mse,1e-9)))),'ssim':float(np.mean(bundle['ssim'][region])),
        'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1_tolerant':ef,'dark_smudge_rate':float(np.sum(bundle['dark_smudge']&region)/max(np.sum(bundle['clean_light']&region),1))}
    if staff_region is not None:out.update(staff_geometry(clean_ink,bundle['recon_ink'],staff_region))
    return out

rows=[];confusion=[];components=[]
for index,page in enumerate(pages,1):
    clean=cv2.imread(str(find_local(LCLEAN,page)),0);observed=cv2.imread(str(find_local(LOBS,page)),0);record=next(r for r in cache_records if r['page']==page)
    with np.load(record['cache']) as z:mask=z['mask'];outputs={m:z[m] for m in METHODS};route_maps={t:{n:z[f'g{t}_{n}']>0 for n in ['preserve','remove','ambiguous']} for t in RAW_THRESHOLDS}
    regions,clean_ink,staff=classify(clean,mask);clean_edges=edges(clean)
    n,labels,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);component_specs=[]
    for i in range(1,n):
        if stats[i,cv2.CC_STAT_AREA]<20:continue
        comp=labels==i;frac=float(np.mean(clean_ink[comp]));category='background' if frac<.05 else ('structure' if frac>.30 else 'mixed');component_specs.append((i,comp,category))
    for method,recon in outputs.items():
        bundle=method_bundle(clean,recon,clean_ink,clean_edges)
        for region_name,region in regions.items():
            q=score(bundle,region,clean_ink,staff if region_name=='staff' else None)
            if q:rows.append({'page':page,'method':method,'region':region_name,**q})
        for i,comp,category in component_specs:
            q=score(bundle,comp,clean_ink,comp&staff)
            if q:components.append({'page':page,'component':i,'category':category,'method':method,'x':int(stats[i,0]),'y':int(stats[i,1]),'w':int(stats[i,2]),'h':int(stats[i,3]),**q})
        del bundle
    for threshold,route in route_maps.items():
        for route_name,route_mask in route.items():
            confusion.append({'page':page,'method':f'geometry_t{threshold}','route':route_name,'pixels':int(route_mask.sum()),'clean_print_pixels':int(np.sum(route_mask&clean_ink)),
                'clean_paper_pixels':int(np.sum(route_mask&~clean_ink)),'print_fraction':float(np.sum(route_mask&clean_ink)/max(route_mask.sum(),1)),
                'exact_observed_copy':bool(np.array_equal(outputs[f'geometry_t{threshold}'][route['preserve']],observed[route['preserve']])) if route_name=='preserve' else np.nan})
    print(f'Scored {index}/{len(pages)} {page}')
results=pd.DataFrame(rows);component_df=pd.DataFrame(components);confusion_df=pd.DataFrame(confusion)
results.to_csv(TABLES/'metrics_by_page_region.csv',index=False);component_df.to_csv(TABLES/'component_metrics.csv',index=False);confusion_df.to_csv(TABLES/'routing_truth_diagnostics.csv',index=False)


## Paired comparisons and hard structural gate


In [ ]:
def bootstrap(values,seed=42):
    values=np.asarray(values,float);values=values[np.isfinite(values)]
    if not len(values):return np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed);samples=np.array([rng.choice(values,len(values),replace=True).mean() for _ in range(BOOTSTRAP_SAMPLES)])
    return float(values.mean()),float(np.percentile(samples,2.5)),float(np.percentile(samples,97.5))
specs=[('background','mae',False),('background','dark_smudge_rate',False),('staff','ink_recall',True),('staff','edge_f1_tolerant',True),
       ('staff','staff_gap_rate',False),('staff','staff_centreline_mae',False),('staff','staff_centreline_p95',False),('staff','staff_thickness_mae',False),
       ('symbol','ink_recall',True),('symbol','edge_f1_tolerant',True),('mixed','edge_f1_tolerant',True),('all','mae',False),('all','ssim',True)]
paired=[]
for region,metric,higher in specs:
    pivot=results[results.region==region].pivot(index='page',columns='method',values=metric)
    if 'telea' not in pivot:continue
    for method in METHODS:
        if method=='telea' or method not in pivot:continue
        delta=(pivot[method]-pivot.telea)*(1 if higher else -1);mean,lo,hi=bootstrap(delta.dropna())
        paired.append({'region':region,'metric':metric,'method':method,'baseline':'telea','benefit_mean':mean,'ci95_low':lo,'ci95_high':hi,'wins':int((delta>0).sum()),'losses':int((delta<0).sum()),'ties':int((delta==0).sum()),'pages':int(delta.notna().sum())})
paired=pd.DataFrame(paired);paired.to_csv(TABLES/'paired_vs_telea.csv',index=False)
summary=results.groupby(['method','region'],as_index=False).agg(pages=('page','nunique'),pixels=('pixels','sum'),mae=('mae','mean'),ssim=('ssim','mean'),ink_recall=('ink_recall','mean'),edge_f1_tolerant=('edge_f1_tolerant','mean'),dark_smudge_rate=('dark_smudge_rate','mean'),staff_gap_rate=('staff_gap_rate','mean'),staff_centreline_mae=('staff_centreline_mae','mean'),staff_centreline_p95=('staff_centreline_p95','mean'),staff_thickness_mae=('staff_thickness_mae','mean'))
summary.to_csv(TABLES/'summary_by_region.csv',index=False)
def value(method,region,metric,column):
    q=paired[(paired.method==method)&(paired.region==region)&(paired.metric==metric)];return float(q.iloc[0][column]) if len(q) else np.nan
route_summary=confusion_df.groupby(['method','route'],as_index=False).agg(pixels=('pixels','sum'),clean_print_pixels=('clean_print_pixels','sum'),clean_paper_pixels=('clean_paper_pixels','sum'))
route_summary['print_fraction']=route_summary.clean_print_pixels/route_summary.pixels.clip(lower=1);route_summary.to_csv(TABLES/'routing_truth_summary.csv',index=False)
gates=[]
for method in ['geometry_t80','geometry_t110']:
    preserved=route_summary[(route_summary.method==method)&(route_summary.route=='preserve')]
    precision=float(preserved.print_fraction.iloc[0]) if len(preserved) else np.nan
    staff_recall_low=value(method,'staff','ink_recall','ci95_low');staff_edge_low=value(method,'staff','edge_f1_tolerant','ci95_low');symbol_edge_low=value(method,'symbol','edge_f1_tolerant','ci95_low');mixed_edge_low=value(method,'mixed','edge_f1_tolerant','ci95_low')
    bg_mae_low=value(method,'background','mae','ci95_low');bg_smudge_low=value(method,'background','dark_smudge_rate','ci95_low')
    structural=bool(staff_recall_low>=0 and staff_edge_low>=0 and symbol_edge_low>=0 and mixed_edge_low>=0)
    gates.append({'method':method,'preserve_route_print_precision':precision,'staff_recall_ci95_low':staff_recall_low,'staff_edge_ci95_low':staff_edge_low,'symbol_edge_ci95_low':symbol_edge_low,'mixed_edge_ci95_low':mixed_edge_low,
        'background_mae_ci95_low':bg_mae_low,'background_smudge_ci95_low':bg_smudge_low,'structural_superiority_supported':structural,'exact_preserve_asserted':True,
        'production_candidate':bool(structural and precision>=0.95 and bg_smudge_low>=-0.01)})
gates=pd.DataFrame(gates);gates.to_csv(TABLES/'structural_gate.csv',index=False);display(gates)


## Contact sheets, route maps and worst structural crops


In [ ]:
priority=component_df[component_df.method=='telea'].sort_values(['staff_centreline_p95','pixels'],ascending=False).page.drop_duplicates().head(MAX_CONTACT_PAGES)
for page in priority:
    clean=cv2.imread(str(find_local(LCLEAN,page)),0);observed=cv2.imread(str(find_local(LOBS,page)),0);record=next(r for r in cache_records if r['page']==page)
    with np.load(record['cache']) as z:
        images=[clean,observed]+[z[m] for m in METHODS];names=['clean','observed']+METHODS
        fig,axes=plt.subplots(2,4,figsize=(24,14))
        for ax,image,name in zip(axes.ravel(),images,names):ax.imshow(image,cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
        plt.tight_layout();plt.savefig(CONTACTS/f'{page}_comparison.png',dpi=150,bbox_inches='tight');plt.close(fig)
        for threshold in RAW_THRESHOLDS:
            rgb=cv2.cvtColor(observed,cv2.COLOR_GRAY2RGB);overlay=rgb.copy();overlay[z[f'g{threshold}_preserve']>0]=(20,160,20);overlay[z[f'g{threshold}_remove']>0]=(30,100,230);overlay[z[f'g{threshold}_ambiguous']>0]=(230,140,20)
            blend=cv2.addWeighted(rgb,.55,overlay,.45,0);cv2.imwrite(str(MAPS/f'{page}_geometry_t{threshold}_routes.png'),cv2.cvtColor(blend,cv2.COLOR_RGB2BGR))
for metric in ['staff_gap_rate','staff_centreline_p95','staff_thickness_mae','edge_f1_tolerant']:
    folder=CROPS/metric;folder.mkdir(parents=True,exist_ok=True)
    ascending=False if metric!='edge_f1_tolerant' else True
    target=component_df[(component_df.category!='background')&(component_df.method=='geometry_t80')].dropna(subset=[metric]).sort_values(metric,ascending=ascending).head(MAX_FAILURE_CROPS)
    for rank,(_,row) in enumerate(target.iterrows(),1):
        page=row.page;clean=cv2.imread(str(find_local(LCLEAN,page)),0);observed=cv2.imread(str(find_local(LOBS,page)),0);record=next(r for r in cache_records if r['page']==page)
        with np.load(record['cache']) as z:images=[clean,observed]+[z[m] for m in METHODS]
        pad=30;H,W=clean.shape;x0=max(0,int(row.x)-pad);y0=max(0,int(row.y)-pad);x1=min(W,int(row.x+row.w)+pad);y1=min(H,int(row.y+row.h)+pad)
        fig,axes=plt.subplots(2,4,figsize=(20,9));names=['clean','observed']+METHODS
        for ax,image,name in zip(axes.ravel(),images,names):ax.imshow(image[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(name);ax.axis('off')
        plt.tight_layout();plt.savefig(folder/f'{rank:03d}_{page}_c{int(row.component)}.png',dpi=170,bbox_inches='tight');plt.close(fig)
print('Contact sheets:',CONTACTS,'Route maps:',MAPS,'Failure crops:',CROPS)


## Manifest


In [ ]:
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.5','schema':SCHEMA,'pages':len(pages),'excluded_pages':sorted(EXCLUDE_PAGES),'dilation':DILATION,'methods':METHODS,
 'settings':{'raw_thresholds':RAW_THRESHOLDS,'paper_threshold':PAPER_THRESHOLD,'normalised_paper_threshold':NORMALISED_PAPER_THRESHOLD,'background_sigma':BACKGROUND_SIGMA,
 'horizontal_length':HORIZONTAL_LENGTH,'vertical_length':VERTICAL_LENGTH,'geometry_grow':GEOMETRY_GROW,'outside_support_radius':OUTSIDE_SUPPORT_RADIUS,
 'compact':[COMPACT_MIN_AREA,COMPACT_MAX_AREA,COMPACT_MAX_DIMENSION,COMPACT_MIN_FILL],'bootstrap_samples':BOOTSTRAP_SAMPLES},
 'guarantees':{'clean_used_for_scoring_and_posthoc_route_diagnostics_only':True,'candidate_functions_do_not_receive_clean':True,'outside_mask_immutable_asserted':True,
 'preserve_route_byte_exact_to_observed_asserted':True,'telea_permitted_only_on_ambiguous_route_for_geometry_candidates':True,'v7_1_dilation2_mask_and_telea_reused':True},
 'limitations':['Genuine pencil/print overlap is unobservable with certainty from an annotated-only scan.','Geometry rules are deterministic hypotheses and require independent confirmation.','Page201 excluded because its historical benchmark mask is known to be incorrect/stale.']}
(OUT/'benchmark_manifest.json').write_text(json.dumps(manifest,indent=2));print('V7.5 COMPLETE:',OUT)
